# Step 2 Optimization: Ocean Cleanup MILP

This notebook converts particle simulation output into a discrete network, solves a **Mixed-Integer Linear Program (MILP)** to maximize plastic intercepted by a cleanup boat, and produces a **Yield Curve** showing optimal launch timing.

**Setup:** Run Step2_Simulation_v3.ipynb first and save `history_x`, `history_y` to a pickle file, or run both in the same session.

## 1. Imports & Configuration

We import the required libraries (numpy, matplotlib, gurobipy) and set all tunable parameters:
- **SIM_DAYS**: Total simulation window (180 days)
- **MISSION_DAYS**: Length of each cleanup mission (60 days)
- **MAX_LAUNCH_DAY**: How far to slide the launch window (0 to 150)
- **CLUSTER_GRID_DEG**: Size of grid cells used to cluster particles (degrees)
- **PORT_LON/LAT**: Home port location (e.g. Singapore)
- **BOAT_***: Speed and fuel/endurance limits

In [1]:
# --- 1. IMPORTS & CONFIGURATION -------------------------------------------------

import pickle
import numpy as np
import matplotlib.pyplot as plt

# Gurobi (pip install gurobipy; requires Gurobi license)
import gurobipy as gp
from gurobipy import GRB

# Simulation window
SIM_DAYS = 180
MISSION_DAYS = 60
MAX_LAUNCH_DAY = 151  # launch days 0..150

# Cluster granularity (degrees)
CLUSTER_GRID_DEG = 1.0

# Port (Node 0) - e.g. Singapore
PORT_LON = 103.85
PORT_LAT = 1.29

# Boat parameters
BOAT_MAX_SPEED_DEG_PER_DAY = 2.0
BOAT_MAX_TOTAL_DISTANCE_DEG = 50.0

print("Configuration loaded.")

ModuleNotFoundError: No module named 'numpy'

## 2. Discretization: Ocean → Network

Before the MILP can run, we turn the continuous particle positions into a discrete network:

- **`haversine_deg`**: Computes distance between two points (in degrees) for feasibility checks.
- **`cluster_particles_by_grid`**: Bins particles into a regular grid. Each non-empty cell becomes one "trash cluster" (node). The reward $R_{it}$ is the number of particles in that cell (plastic density).
- **`build_daily_network`**: Loops over each day of the simulation, clusters that day's particles, and returns `clusters_t` (list of cluster centers per day) and `rewards_t` (list of rewards per cluster per day).

In [ ]:
# --- 2. DISCRETIZATION: Ocean → Network -----------------------------------------

def haversine_deg(lon1, lat1, lon2, lat2):
    """Approximate distance in degrees."""
    return np.sqrt((lon2 - lon1) ** 2 + (lat2 - lat1) ** 2)


def cluster_particles_by_grid(x, y, grid_deg=CLUSTER_GRID_DEG, min_particles=1):
    """Group particles into grid cells. Each non-empty cell = cluster."""
    if len(x) == 0:
        return [], []
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
    ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
    xmin, xmax = max(xmin, -180), min(xmax, 180)
    ymin, ymax = max(ymin, -90), min(ymax, 90)

    xb = np.floor((x - xmin) / grid_deg).astype(int)
    yb = np.floor((y - ymin) / grid_deg).astype(int)
    valid = np.isfinite(x) & np.isfinite(y)
    xb, yb = xb[valid], yb[valid]
    xv, yv = x[valid], y[valid]

    cells = {}
    for i in range(len(xb)):
        key = (xb[i], yb[i])
        if key not in cells:
            cells[key] = []
        cells[key].append((float(xv[i]), float(yv[i])))

    centers, rewards = [], []
    for (xi, yi), pts in cells.items():
        if len(pts) < min_particles:
            continue
        cx = sum(p[0] for p in pts) / len(pts)
        cy = sum(p[1] for p in pts) / len(pts)
        centers.append((cx, cy))
        rewards.append(float(len(pts)))
    return centers, rewards


def build_daily_network(history_x, history_y, sim_days=SIM_DAYS, grid_deg=CLUSTER_GRID_DEG):
    """For each day t: cluster particles into nodes, assign rewards R_it."""
    clusters_t = []
    rewards_t = []
    n_days = min(sim_days, len(history_x))
    for t in range(n_days):
        x = np.asarray(history_x[t], dtype=float)
        y = np.asarray(history_y[t], dtype=float)
        centers, R = cluster_particles_by_grid(x, y, grid_deg=grid_deg)
        clusters_t.append(centers)
        rewards_t.append(R)
    return clusters_t, rewards_t

print("Discretization functions defined.")

## 3. MILP Solver (Gurobi)

`solve_mission_milp` implements the optimization model:

**Decision variables:**
- **$x_{t,j,i}$** (or port→j): Binary. 1 if the boat travels to cluster $j$ on day $t$ (from cluster $i$ on day $t-1$, or from port if $t=0$).
- **$y_{t,j}$**: Binary. 1 if the boat harvests (cleans) cluster $j$ on day $t$.

**Objective:** Maximize total plastic intercepted: $\sum_{t,j} R_{t,j} \cdot y_{t,j}$

**Constraints:**
- **Origin rule**: Boat leaves port at most once on day 0.
- **Visit implies harvest**: Can only harvest a cluster if the boat visits it.
- **Flow / single destination per day**: At most one destination per day.
- **Fuel limit**: Total distance traveled ≤ `BOAT_MAX_TOTAL_DISTANCE_DEG`.

In [ ]:
# --- 3. MILP SOLVER (Gurobi) ----------------------------------------------------

def solve_mission_milp(clusters_t, rewards_t, start_day, mission_days, port,
                       max_speed_deg=BOAT_MAX_SPEED_DEG_PER_DAY,
                       max_total_distance_deg=BOAT_MAX_TOTAL_DISTANCE_DEG):
    """
    Solve cleanup MILP for mission window [start_day, start_day + mission_days).
    Uses Gurobi. Returns (total_reward, info_dict or None).
    """
    end_day = start_day + mission_days
    clusters_window = clusters_t[start_day:end_day]
    rewards_window = rewards_t[start_day:end_day]
    T = len(clusters_window)
    if T == 0:
        return 0.0, None

    port_lon, port_lat = port

    def dist_pts(p1, p2):
        return haversine_deg(p1[0], p1[1], p2[0], p2[1])

    N = [len(clusters_window[t]) for t in range(T)]
    pos = [clusters_window[t] for t in range(T)]
    R = [rewards_window[t] for t in range(T)]

    m = gp.Model("CleanupMission")
    m.setParam("OutputFlag", 0)

    x = {}
    y = {}
    for t in range(T):
        y[t] = {j: m.addVar(vtype=GRB.BINARY, name=f"y_{t}_{j}") for j in range(N[t])}
        x[t] = {}
        for j in range(N[t]):
            x[t][j] = {}
            if t == 0:
                if dist_pts((port_lon, port_lat), pos[t][j]) <= max_speed_deg:
                    x[t][j][-1] = m.addVar(vtype=GRB.BINARY, name=f"x_{t}_{j}_port")
            if t >= 1:
                for i in range(N[t - 1]):
                    if dist_pts(pos[t - 1][i], pos[t][j]) <= max_speed_deg:
                        x[t][j][i] = m.addVar(vtype=GRB.BINARY, name=f"x_{t}_{j}_{i}")

    m.setObjective(
        gp.quicksum(R[t][j] * y[t][j] for t in range(T) for j in range(N[t])),
        GRB.MAXIMIZE,
    )

    m.addConstr(
        gp.quicksum(x[0][j][-1] for j in range(N[0]) if -1 in x[0][j]) <= 1,
        name="leave_port",
    )

    for t in range(T):
        for j in range(N[t]):
            in_terms = []
            if t == 0 and -1 in x[0][j]:
                in_terms.append(x[0][j][-1])
            if t >= 1:
                in_terms.extend(x[t][j][i] for i in range(N[t - 1]) if i in x[t][j])
            in_flow = gp.quicksum(in_terms) if in_terms else 0
            m.addConstr(y[t][j] <= in_flow, name=f"harvest_if_visit_{t}_{j}")
            if in_terms:
                m.addConstr(in_flow <= 1, name=f"visit_once_{t}_{j}")

    for t in range(T):
        m.addConstr(
            gp.quicksum(x[t][j][i] for j in range(N[t]) for i in x[t][j]) <= 1,
            name=f"one_dest_day_{t}",
        )

    dist_expr = gp.LinExpr()
    for t in range(T):
        for j in range(N[t]):
            if t == 0 and -1 in x[0].get(j, {}):
                dist_expr += dist_pts((port_lon, port_lat), pos[t][j]) * x[0][j][-1]
            if t >= 1:
                for i in range(N[t - 1]):
                    if i in x[t].get(j, {}):
                        dist_expr += dist_pts(pos[t - 1][i], pos[t][j]) * x[t][j][i]
    m.addConstr(dist_expr <= max_total_distance_deg, name="fuel_limit")

    m.optimize()
    if m.Status not in (GRB.OPTIMAL, GRB.TIME_LIMIT) or m.SolCount == 0:
        return 0.0, None

    total_reward = m.ObjVal
    harvests = [(t, j) for t in range(T) for j in range(N[t]) if y[t][j].X > 0.5]
    info = {"harvests": harvests, "pos": pos, "clusters_window": clusters_window}
    return float(total_reward), info

print("MILP solver ready (Gurobi). Install: pip install gurobipy")

## 4. Yield Curve: Sliding Window Analytics

Instead of solving one long mission, we slide a 60-day window and solve a new MILP for each possible launch day:

- **`run_yield_curve`**: For launch days 0 to 150, solves the MILP and records the maximum plastic intercepted. Returns (launch_days, plastic) for plotting.
- **`plot_yield_curve`**: Plots Launch Day (x-axis) vs Maximum Plastic Intercepted (y-axis). Peaks indicate the best times to deploy; valleys indicate when launching would be inefficient.

In [ ]:
# --- 4. YIELD CURVE: Sliding Window Analytics -----------------------------------

def run_yield_curve(clusters_t, rewards_t, mission_days=MISSION_DAYS, max_launch_day=MAX_LAUNCH_DAY, port=(PORT_LON, PORT_LAT)):
    """Run MILP for each launch day 0..max_launch_day-1. Returns (launch_days, plastic)."""
    launch_days = []
    plastic = []
    for start in range(max_launch_day):
        end = start + mission_days
        if end > len(clusters_t):
            break
        reward, _ = solve_mission_milp(clusters_t, rewards_t, start_day=start, mission_days=mission_days, port=port)
        launch_days.append(start)
        plastic.append(reward)
    return launch_days, plastic


def plot_yield_curve(launch_days, plastic, title="Yield Curve: Optimal Launch Timing", save_path=None):
    """Plot Launch Day vs Maximum Plastic Intercepted."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(launch_days, plastic, "b-", linewidth=2, label="Max plastic intercepted")
    ax.fill_between(launch_days, plastic, alpha=0.3)
    ax.set_xlabel("Launch Day")
    ax.set_ylabel("Maximum Plastic Intercepted (particles)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")
    plt.show()

print("Yield curve functions defined.")

## 5. Load Simulation Data & Build Network

Load particle trajectories from the Step2 simulation:

- **Option A (pickle)**: Save `history_x`, `history_y` in Step2 with `pickle.dump(...)`, then load here.
- **Option B (in-memory)**: Run Step2 first in the same kernel and use its `history_x`, `history_y`.
- **Demo fallback**: If neither is available, synthetic particle data is generated so the notebook runs end-to-end.

In [ ]:
# --- 5. LOAD SIMULATION & RUN -----------------------------------------------

# Option A: Load from pickle (uncomment and set path)
# with open("sim_output.pkl", "rb") as f:
#     data = pickle.load(f)
# history_x, history_y = data["history_x"], data["history_y"]

# Option B: Use in-memory from Step2 (run Step2 first, then this notebook)
# history_x, history_y = ...  # from Step2_Simulation_v3

# For demo: generate synthetic data if no sim output available
try:
    history_x
    history_y
except NameError:
    print("No history_x/history_y found. Generating synthetic demo data...")
    np.random.seed(42)
    history_x = [90 + 40 * np.random.rand(500 + d * 20) for d in range(SIM_DAYS)]
    history_y = [-10 + 35 * np.random.rand(500 + d * 20) for d in range(SIM_DAYS)]

print("1. Building daily cluster network...")
clusters_t, rewards_t = build_daily_network(history_x, history_y)
print(f"   Days: {len(clusters_t)}, Total clusters: {sum(len(c) for c in clusters_t)}")

## 6. Run Yield Curve & Plot Results

Run the sliding-window optimization loop, identify the peak launch day, and plot the Yield Curve. The plot shows which launch dates maximize plastic interception over the mission window.

In [ ]:
print("2. Running Yield Curve (sliding 60-day MILP)...")
launch_days, plastic = run_yield_curve(clusters_t, rewards_t)

peak_day = launch_days[np.argmax(plastic)]
peak_yield = max(plastic)
print(f"   Peak launch day: {peak_day} (plastic intercepted: {peak_yield:.0f})")

print("3. Plotting Yield Curve...")
plot_yield_curve(
    launch_days, plastic,
    title=f"Yield Curve: Launch Day vs Plastic Intercepted (Peak: Day {peak_day})",
    save_path="yield_curve.png",
)